## Building Baseline Models And Generating `leaderboard.csv`

In this tutorial, we will continue from the last two notebooks and use our preprocessed **MovieLens 1M dataset** to build baseline models and generating the `leaderboard.csv`.

### What We're Achieving Here:

Through this notebook we will implement the following two baselines approaches for our recommendation system:
- **popularity@session** – globally popular items excluding user history
- **als@factors=64** – implicit matrix factorization

Next, we will:
- Evaluate baselines using the project’s metric harness.
- Aggregate results into a unified `leaderboard.csv` file.

### Key-Takeaways:
- Implement simple, reproducible recommendation baselines.
- Understand the purpose of baselines in a recsys pipeline.
- Produce a unified **leaderboard.csv** summarizing val/test performance for both baselines.

### Available Dataset Files

From metadata, the project includes two complete split sets:
- `split_metadata_20251103_1426.json`
- `split_metadata_20251103_1428.json`
- `train_20251103_1426.csv`, `valid_20251103_1426.csv`, `test_20251103_1426.csv`
- `train_20251103_1428.csv`, `valid_20251103_1428.csv`, `test_20251103_1428.csv`


Each split has the following columns:
```
UserID, MovieID, Rating, Timestamp, Datetime, label
```

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import json
from pathlib import Path

base_path = (
    "/content/drive/MyDrive/upgrad_live_sessions/"
    "Recommendation_systems/notebook-1/C6/data_splits/"
)
# base_path = "/content/"
meta_1426 = json.load(open(base_path + 'split_metadata_20251103_1426.json'))
meta_1428 = json.load(open(base_path + 'split_metadata_20251103_1428.json'))


meta_1426, meta_1428

({'timestamp': '20251103_1426',
  'total_interactions': 999917,
  'train_rows': 987837,
  'val_rows': 6040,
  'test_rows': 6040,
  'n_users': 6040,
  'n_items': 3503,
  'rating_threshold': 4.0,
  'val_k': 1,
  'test_k': 1},
 {'timestamp': '20251103_1428',
  'total_interactions': 999917,
  'train_rows': 987837,
  'val_rows': 6040,
  'test_rows': 6040,
  'n_users': 6040,
  'n_items': 3503,
  'rating_threshold': 4.0,
  'val_k': 1,
  'test_k': 1})

### Load Train / Valid / Test Splits

In [3]:
import pandas as pd

train = pd.read_csv(base_path + 'train_20251103_1426.csv')
valid = pd.read_csv(base_path + 'val_20251103_1426.csv')
test = pd.read_csv(base_path + 'test_20251103_1426.csv')


display(train.head())
display(valid.head())
display(test.head())

,UserID,MovieID,Rating,Timestamp,Datetime,label
0,1,3186,4,978300019,2000-12-31 22:00:19+00:00,True
1,1,1270,5,978300055,2000-12-31 22:00:55+00:00,True
2,1,1721,4,978300055,2000-12-31 22:00:55+00:00,True
3,1,1022,5,978300055,2000-12-31 22:00:55+00:00,True
4,1,2340,3,978300103,2000-12-31 22:01:43+00:00,False


,UserID,MovieID,Rating,Timestamp,Datetime,label,cold_user,cold_item
0,1,1907,4,978824330,2001-01-06 23:38:50+00:00,True,False,False
1,2,1544,4,978300174,2000-12-31 22:02:54+00:00,True,False,False
2,3,3868,3,978298486,2000-12-31 21:34:46+00:00,False,False,False
3,4,1036,4,978294282,2000-12-31 20:24:42+00:00,True,False,False
4,5,1884,3,978246576,2000-12-31 07:09:36+00:00,False,False,False


,UserID,MovieID,Rating,Timestamp,Datetime,label,cold_user,cold_item
0,1,48,5,978824351,2001-01-06 23:39:11+00:00,True,False,False
1,2,1917,3,978300174,2000-12-31 22:02:54+00:00,False,False,False
2,3,2081,4,978298504,2000-12-31 21:35:04+00:00,True,False,False
3,4,1954,5,978294282,2000-12-31 20:24:42+00:00,True,False,False
4,5,288,2,978246585,2000-12-31 07:09:45+00:00,False,False,False


### Preprocess Interaction Columns

Let us do some column renaming for the dataframe headers according to Python PEP8 guidelines. We will rename the field names to a generic style: `user_id`, `item_id`, `rating`, `timestamp`.

In [4]:
train = train.rename(columns={'UserID':'user_id','MovieID':'item_id','Rating':'rating','Timestamp':'timestamp'})
valid = valid.rename(columns={'UserID':'user_id','MovieID':'item_id','Rating':'rating','Timestamp':'timestamp'})
test = test.rename(columns={'UserID':'user_id','MovieID':'item_id','Rating':'rating','Timestamp':'timestamp'})

train.shape, valid.shape, test.shape

((987837, 6), (6040, 8), (6040, 8))

### Baseline 1 — popularity@session

Next, we will create our first baseline model using `popularity@session` approach

The approach `popularity@session` recommends the globally most popular items (from *train*), skipping items a user has already interacted with.

In [5]:
from collections import Counter, defaultdict


# Compute global popularity
pop_counts = Counter(train['item_id'])
popular_items = [it for it,_ in pop_counts.most_common()]

In [6]:
# Build user histories
user_history = defaultdict(set)
for _, row in train.iterrows():
  user_history[row['user_id']].add(row['item_id'])

In [7]:
# Recommend top-K
def rec_popularity_session(user_id, K):
  seen = user_history[user_id]
  out = []
  for item in popular_items:
    if item not in seen:
      out.append(item)
    if len(out) >= K:
      break
  return out

We will apply these models a bit later; next let us build the second approach.

### Baseline 2 — ALS@factors=64
Alternating Least Squares (ALS) algorithm used in the context of matrix factorization for recommendation systems, where "64" might denote a specific implementation detail, such as the number of latent factors.

ALS is a popular, model-based collaborative filtering algorithm. It operates on the principle that past user behavior can predict future preferences: if two users liked similar items in the past, they are likely to like similar items in the future.

The primary goal of ALS is to address the data sparsity problem inherent in user-item rating matrices (where most users have only rated a small fraction of available items).

We will implement ALS@factor=64 using the **implicit** library.

In [8]:
!pip install implicit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 45.3 MB/s eta 0:00:00


In [9]:
import scipy.sparse as sps
import implicit

In [10]:
# Map users/items to internal indices
# Build user and item index maps in FIRST APPEARANCE ORDER
user2idx = {}
item2idx = {}

user_list = []
item_list = []

row_idx = []
col_idx = []
data = []

for _, row in train.iterrows():
    u = row["user_id"]
    i = row["item_id"]

    if u not in user2idx:
        user2idx[u] = len(user2idx)
        user_list.append(u)

    if i not in item2idx:
        item2idx[i] = len(item2idx)
        item_list.append(i)

    row_idx.append(user2idx[u])
    col_idx.append(item2idx[i])
    data.append(1.0)

In [11]:
# Build user–item matrix
n_users = len(user2idx)
n_items = len(item2idx)

mat = sps.csr_matrix((data, (row_idx, col_idx)), shape=(n_users, n_items))

In [12]:
# Train ALS (item-user matrix)
als = implicit.als.AlternatingLeastSquares(
    factors=64,
    regularization=0.01,
    iterations=20
)
als.fit(mat)

print(f"User factors shape: {als.user_factors.shape}")  # Should be (6040, 64)
print(f"Item factors shape: {als.item_factors.shape}") # Should be (3503, 64)

/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

User factors shape: (6040, 64)
Item factors shape: (3503, 64)


In [13]:
# item index reverse map
idx2item = {v: k for k, v in item2idx.items()}

In [14]:
def normalize_recs(recs):
    """Normalize implicit recommend outputs into list of (item, score)."""
    if recs is None:
        return []
    # tuple of arrays (ids, scores)
    if isinstance(recs, tuple) and len(recs) == 2:
        return list(zip(recs[0], recs[1]))
    arr = np.asarray(recs)
    if arr.ndim == 2 and arr.shape[1] >= 2:
        return list(zip(arr[:,0].astype(int), arr[:,1].astype(float)))
    try:
        return [(int(x[0]), float(x[1])) for x in recs]
    except:
        return []

In [15]:
def rec_als(user_id, K=10):
    # Check if user exists in training data
    if user_id not in user2idx:
        return []

    uidx = user2idx[user_id]

    # Double-check bounds
    if uidx >= mat.shape[0] or uidx >= als.user_factors.shape[0]:
        return []

    user_items_row = mat[uidx:uidx+1]

    try:
        raw = als.recommend(
            userid=uidx,
            user_items=user_items_row,
            N=min(K*2, mat.shape[1]),  # Don't request more items than exist
            filter_already_liked_items=True,
            recalculate_user=False
        )
    except (IndexError, ValueError) as e:
        print(f"Error for user {user_id} (index {uidx}): {e}")
        return []

    parsed = normalize_recs(raw)

    out = []
    for internal_id, score in parsed:
        if internal_id not in idx2item:
            continue

        ext = idx2item[internal_id]
        if ext not in user_history[user_id]:
            out.append(ext)
        if len(out) >= K:
            break

    return out

### Metric Functions (Precision, Recall, NDCG)

Let us get the metric functions as seen before in the previous notebooks

In [16]:
import numpy as np

def prn_at_k(recs, gt, K):
  hits = [1 if r in gt else 0 for r in recs[:K]]
  precision = sum(hits)/K
  recall = sum(hits)/len(gt) if gt else 0
  dcg = sum(h/np.log2(i+2) for i,h in enumerate(hits))
  ideal = min(len(gt),K)
  idcg = sum(1/np.log2(i+2) for i in range(ideal))
  ndcg = dcg/idcg if idcg>0 else 0
  return precision, recall, ndcg

### Evaluation Harness (as in Notebook 2)

In [17]:
Ks = [5,10,20]


def evaluate(df, rec_fn):
  results = {k:{'precision':0,'recall':0,'ndcg':0,'n':0} for k in Ks}
  for user, grp in df.groupby('user_id'):
    gt = set(grp['item_id'])
    for k in Ks:
      recs = rec_fn(user, k)
      p,r,n = prn_at_k(recs, gt, k)
      results[k]['precision'] += p
      results[k]['recall'] += r
      results[k]['ndcg'] += n
      results[k]['n'] += 1
  final = {
    'precision':{k:results[k]['precision']/results[k]['n'] for k in Ks},
    'recall':{k:results[k]['recall']/results[k]['n'] for k in Ks},
    'ndcg':{k:results[k]['ndcg']/results[k]['n'] for k in Ks},
    'n_users_eval':{k:results[k]['n'] for k in Ks},
  }
  return final

### Run Baselines

In [18]:
val_pop = evaluate(valid, rec_popularity_session)
test_pop = evaluate(test, rec_popularity_session)

val_als = evaluate(valid, rec_als)
test_als = evaluate(test, rec_als)

val_pop, test_pop

({'precision': {5: 0.0042384105960264805,
   10: 0.004271523178807963,
   20: 0.0035347682119205577},
  'recall': {5: 0.02119205298013245,
   10: 0.04271523178807947,
   20: 0.0706953642384106},
  'ndcg': {5: np.float64(0.01378625099035152),
   10: np.float64(0.020597701220021452),
   20: np.float64(0.027623412169896402)},
  'n_users_eval': {5: 6040, 10: 6040, 20: 6040}},
 {'precision': {5: 0.003973509933774826,
   10: 0.0036754966887417293,
   20: 0.0033857615894039993},
  'recall': {5: 0.019867549668874173,
   10: 0.036754966887417216,
   20: 0.06771523178807946},
  'ndcg': {5: np.float64(0.012554130354386966),
   10: np.float64(0.01794165905760328),
   20: np.float64(0.025662214447899238)},
  'n_users_eval': {5: 6040, 10: 6040, 20: 6040}})

### Build `leaderboard.csv`

In [19]:
rows = []

In [20]:
def add_rows(model, notes, split, metrics):
  for k in Ks:
    rows.append({
      'model':model,
      'notes':notes,
      'split':split,
      'K':k,
      'Precision':metrics['precision'][k],
      'Recall':metrics['recall'][k],
      'NDCG':metrics['ndcg'][k],
      'UsersEval':metrics['n_users_eval'][k]
    })

In [21]:
add_rows('popularity@session','global popularity baseline','val',val_pop)
add_rows('popularity@session','global popularity baseline','test',test_pop)
add_rows('als@factors=64','implicit ALS','val',val_als)
add_rows('als@factors=64','implicit ALS','test',test_als)

In [22]:
leaderboard = pd.DataFrame(rows)
leaderboard.to_csv('leaderboard.csv', index=False)
display(leaderboard)

,model,notes,split,K,Precision,Recall,NDCG,UsersEval
0,popularity@session,global popularity baseline,val,5,0.004238,0.021192,0.013786,6040
1,popularity@session,global popularity baseline,val,10,0.004272,0.042715,0.020598,6040
2,popularity@session,global popularity baseline,val,20,0.003535,0.070695,0.027623,6040
3,popularity@session,global popularity baseline,test,5,0.003974,0.019868,0.012554,6040
4,popularity@session,global popularity baseline,test,10,0.003675,0.036755,0.017942,6040
5,popularity@session,global popularity baseline,test,20,0.003386,0.067715,0.025662,6040
6,als@factors=64,implicit ALS,val,5,0.010728,0.053642,0.032147,6040
7,als@factors=64,implicit ALS,val,10,0.009238,0.092384,0.044676,6040
8,als@factors=64,implicit ALS,val,20,0.008204,0.164073,0.062672,6040
9,als@factors=64,implicit ALS,test,5,0.010265,0.051325,0.031388,6040


### Conclusion

In this notebook, we built two core baselines—**popularity@session** and **als@factors=64**—following the same dataset structure and evaluation process established in Notebooks 1 and 2. Using the temporal splits provided (`train_20251103_1426.csv`, `valid_20251103_1426.csv`, `test_20251103_1426.csv`), we computed Precision@K, Recall@K, and NDCG@K for both baselines on validation and test sets. These results were aggregated into a unified `leaderboard.csv`, forming the foundation for comparing future, more advanced recommendation models.